In [1]:
import os
import gc
from pathlib import Path
import pyarrow
import fastparquet
from statsmodels.tsa.seasonal import seasonal_decompose
import missingno as msno
import numpy as np
import pandas as pd
import pyreadstat
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
RUTA_DATA = Path("../data/analytical/mortalidad_raw_slim.parquet")
dfmortalidad = pd.read_parquet(RUTA_DATA)

In [3]:
dfmortalidad.head()

,Anio_Defuncion,Mes_Defuncion,Departamento_Residencia,Sexo,Grupo_Edad_Detallado,Nivel_Educativo,Regimen_Salud,Area_Residencia,Sitio_Defuncion,Asistencia_Medica
0,2008,2,68,2,19,2,1,1,1,1
1,2008,2,68,1,18,2,3,1,2,1
2,2008,2,68,1,20,3,2,1,2,1
3,2008,2,68,2,20,5,2,1,1,1
4,2008,1,54,1,22,2,2,1,1,1


In [13]:
RUTA_PANEL = Path("../data/analytical/panel_dpto_año.parquet")
df_panel = pd.read_parquet(RUTA_PANEL)
df_panel.head()

,cod_dpto,año,departamento,poblacion_total,poblacion_hombre,poblacion_mujer,muertes_total,muertes_hombre,muertes_mujer,muertes_urbano,muertes_rural,tasa_cruda,tasa_ajustada_edad,irca_global,irca_urbano,irca_rural,nivel_riesgo_agua,prevalencia_tabaco
0,5,2008,Antioquia,5662099,2730505,2931594,603.0,348.0,255.0,507.0,96.0,10.649761,12.029387,4.4,4.4,4.7,Sin riesgo,31.1
1,5,2009,Antioquia,5730377,2764235,2966142,585.0,330.0,255.0,476.0,109.0,10.208752,11.200228,5.4,5.5,5.0,Riesgo bajo,31.1
2,5,2010,Antioquia,5800160,2798757,3001403,596.0,359.0,237.0,496.0,100.0,10.275579,11.130002,16.5,10.4,31.0,Riesgo medio,31.1
3,5,2011,Antioquia,5869191,2833245,3035946,569.0,329.0,240.0,482.0,87.0,9.694692,10.128874,20.9,13.3,36.5,Riesgo medio,31.1
4,5,2012,Antioquia,5935916,2866567,3069349,599.0,344.0,255.0,483.0,116.0,10.091113,10.462913,7.5,7.5,7.1,Riesgo bajo,31.1


In [14]:
dfmortalidad.info()

<class 'pandas.DataFrame'>
RangeIndex: 85043 entries, 0 to 85042
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   Anio_Defuncion           85043 non-null  int16   
 1   Mes_Defuncion            85043 non-null  category
 2   Departamento_Residencia  85043 non-null  category
 3   Sexo                     85043 non-null  category
 4   Grupo_Edad_Detallado     85043 non-null  category
 5   Nivel_Educativo          85043 non-null  category
 6   Regimen_Salud            85043 non-null  category
 7   Area_Residencia          85043 non-null  category
 8   Sitio_Defuncion          85043 non-null  category
 9   Asistencia_Medica        85043 non-null  category
dtypes: category(9), int16(1)
memory usage: 914.7 KB


In [6]:
df_panel.info()

<class 'pandas.DataFrame'>
RangeIndex: 561 entries, 0 to 560
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   cod_dpto            561 non-null    int64  
 1   año                 561 non-null    int64  
 2   departamento        561 non-null    str    
 3   poblacion_total     561 non-null    int64  
 4   poblacion_hombre    561 non-null    int64  
 5   poblacion_mujer     561 non-null    int64  
 6   muertes_total       561 non-null    float64
 7   muertes_hombre      561 non-null    float64
 8   muertes_mujer       561 non-null    float64
 9   muertes_urbano      561 non-null    float64
 10  muertes_rural       561 non-null    float64
 11  tasa_cruda          561 non-null    float64
 12  tasa_ajustada_edad  561 non-null    float64
 13  irca_global         537 non-null    float64
 14  irca_urbano         537 non-null    str    
 15  irca_rural          537 non-null    str    
 16  nivel_riesgo_agua  

In [11]:
#total muertes
total_muertes = df_panel['muertes_total'].sum()
print(f"Total de muertes: {total_muertes}")

Total de muertes: 84989.0


In [18]:
#valores únicos en la columna 'Departamento_Residencia'
departamentos_unicos = dfmortalidad['Departamento_Residencia'].unique()
print("Departamentos únicos en dfmortalidad:")
print(departamentos_unicos)

Departamentos únicos en dfmortalidad:
['68', '54', '1', '81', '20', ..., '91', '41', '99', '97', '94']
Length: 34
Categories (34, str): ['1', '11', '13', '15', ..., '94', '95', '97', '99']


In [15]:
# Porque el total de muertes es diferente al sumar la columna 'muertes_total' del df_panel?
#verificar la cantidad de filas vs la cantidad de muertes
total_filas = dfmortalidad.shape[0]

print(f"Total de filas en dfmortalidad: {total_filas}")
print(f"Total de muertes en dfmortalidad: {total_muertes}")

Total de filas en dfmortalidad: 85043
Total de muertes en dfmortalidad: 84989.0
